# AgriLink — Person 3: Collaborative Filtering Recommendation System

**Purpose:** Generate personalized crop recommendations using customer-product interaction history and User-Based Collaborative Filtering.

This notebook covers **only Person 3 — Collaborative Filtering**:
1. Load/create interaction data
2. Create interaction scores
3. Create customer-item matrix
4. Calculate customer similarity using cosine similarity
5. Find similar customers
6. Recommend products purchased by similar customers
7. Produce the common integration output:
   `customer_id, listing_id, crop_id, crop_name, score, recommendation_type, reason`

The interaction-score idea follows the project specification:
- VIEW = 1
- SEARCH = 1
- WISHLIST = 2
- ADD_TO_CART = 3
- RATING = 4
- PURCHASE = 5

Synthetic interaction data is used here for development/testing. It can later be replaced by the real `user_interaction`, `customer_order`, `order_item`, `seller_listing`, and `crop` data.

In [ ]:
# COMMAND: Install required libraries
!pip install -q pandas numpy scikit-learn supabase

In [ ]:
# COMMAND: Import libraries

import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity

import random
import warnings

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

## 1. Crop Catalogue

The current testing catalogue contains 15 agricultural crops.

In [ ]:
# COMMAND: Define crop catalogue

crops = [
    "Apple",
    "Banana",
    "Beans",
    "Brinjal",
    "Cabbage",
    "Carrot",
    "Coriander",
    "Cucumber",
    "Green Chilli",
    "Onion",
    "Orange",
    "Papaya",
    "Potato",
    "Rice",
    "Tomato"
]

print("Number of crops:", len(crops))
print("Crops:")
print(", ".join(crops))

## 2. Define Interaction Scores

The project specification assigns different scores to different customer interactions.

A purchase is given the highest score because it represents the strongest direct purchase signal in this MVP.

In [ ]:
# COMMAND: Define interaction weights

INTERACTION_WEIGHTS = {
    "VIEW": 1,
    "SEARCH": 1,
    "WISHLIST": 2,
    "ADD_TO_CART": 3,
    "RATING": 4,
    "PURCHASE": 5
}

pd.DataFrame(
    list(INTERACTION_WEIGHTS.items()),
    columns=["interaction_type", "score"]
)

## 3. Generate Synthetic User Interaction Data

For testing, each customer interacts with multiple crops.

The generated dataset contains:
- `customer_id`
- `crop_name`
- `interaction_type`
- `interaction_score`

Repeated interactions for the same customer and crop will later be aggregated into one customer-product score.

In [ ]:
# COMMAND: Generate synthetic interaction data

random.seed(42)
np.random.seed(42)

NUM_CUSTOMERS = 1000

records = []

for customer_id in range(1, NUM_CUSTOMERS + 1):

    # Each customer interacts with 3 to 10 crops
    num_products = random.randint(3, 10)

    selected_products = random.sample(
        crops,
        num_products
    )

    for product in selected_products:

        # Generate 1 to 4 interactions for each selected crop
        num_interactions = random.randint(1, 4)

        for _ in range(num_interactions):

            interaction_type = random.choices(
                list(INTERACTION_WEIGHTS.keys()),
                weights=[20, 10, 10, 15, 5, 40],
                k=1
            )[0]

            records.append({
                "customer_id": customer_id,
                "crop_name": product,
                "interaction_type": interaction_type,
                "interaction_score": INTERACTION_WEIGHTS[
                    interaction_type
                ]
            })

interaction_data = pd.DataFrame(records)

print("Interaction dataset created.")
print("Rows:", len(interaction_data))
print("Customers:", interaction_data["customer_id"].nunique())
print("Crops:", interaction_data["crop_name"].nunique())

display(interaction_data.head(20))

In [ ]:
# COMMAND: Validate interaction data

print("Missing values:")
display(interaction_data.isnull().sum().to_frame("missing_values"))

print("\nInteraction counts:")
display(
    interaction_data["interaction_type"]
    .value_counts()
    .to_frame("count")
)

print("\nData types:")
display(interaction_data.dtypes.to_frame("dtype"))

## 4. Create Customer-Product Interaction Scores

If a customer interacts with the same crop several times, the scores are summed.

For example:

Customer 1 + Tomato:
- VIEW = 1
- ADD_TO_CART = 3
- PURCHASE = 5

Total interaction score = 9

This gives us a numerical representation of customer preference.

In [ ]:
# COMMAND: Aggregate interaction scores

customer_product_scores = (
    interaction_data
    .groupby(
        ["customer_id", "crop_name"],
        as_index=False
    )["interaction_score"]
    .sum()
)

print("Customer-product score rows:",
      len(customer_product_scores))

display(customer_product_scores.head(20))

## 5. Create Customer-Item Matrix

Rows = customers

Columns = crops

Values = total interaction score

This is the matrix on which customer-to-customer cosine similarity is calculated.

In [ ]:
# COMMAND: Create customer-item matrix

customer_item_matrix = (
    customer_product_scores
    .pivot_table(
        index="customer_id",
        columns="crop_name",
        values="interaction_score",
        aggfunc="sum",
        fill_value=0
    )
)

# Ensure all catalogue crops are represented
customer_item_matrix = customer_item_matrix.reindex(
    columns=crops,
    fill_value=0
)

customer_item_matrix = customer_item_matrix.astype(float)

print("Customer-item matrix shape:",
      customer_item_matrix.shape)

display(customer_item_matrix.head(10))

In [ ]:
# COMMAND: Inspect one customer's interaction vector

customer_id = 1

customer_vector = customer_item_matrix.loc[
    customer_id
]

display(
    customer_vector[
        customer_vector > 0
    ].sort_values(
        ascending=False
    ).to_frame("interaction_score")
)

## 6. Calculate Customer Similarity

Cosine similarity compares customer interaction vectors.

Customers with similar interaction patterns receive a higher similarity score.

The result is a customer × customer similarity matrix.

In [ ]:
# COMMAND: Calculate customer similarity

similarity_matrix = cosine_similarity(
    customer_item_matrix
)

customer_similarity = pd.DataFrame(
    similarity_matrix,
    index=customer_item_matrix.index,
    columns=customer_item_matrix.index
)

print("Similarity matrix shape:",
      customer_similarity.shape)

display(customer_similarity.iloc[:10, :10])

## 7. Find Similar Customers

For a target customer, the target customer itself must be excluded.

The remaining customers are sorted by similarity score, and the top customers are treated as the most similar users.

In [ ]:
# COMMAND: Find most similar customers

def get_similar_customers(customer_id, top_n=10):

    if customer_id not in customer_similarity.index:
        return pd.DataFrame(
            columns=[
                "customer_id",
                "similarity"
            ]
        )

    similarities = (
        customer_similarity
        .loc[customer_id]
        .drop(index=customer_id)
        .sort_values(
            ascending=False
        )
        .head(top_n)
    )

    result = similarities.reset_index()

    result.columns = [
        "customer_id",
        "similarity"
    ]

    return result

similar_customers = get_similar_customers(
    customer_id=1,
    top_n=10
)

display(similar_customers)

## 8. Generate Collaborative Filtering Recommendations

Recommendation logic:

`Target customer → Similar customers → Their highly interacted crops → Remove crops already interacted with → Rank candidates`

The recommendation score is calculated from the similarity of the customers and the interaction strength of the candidate product.

For this MVP:

`score = similarity × interaction_score`

This is a project-defined ranking score, not a probability.

In [ ]:
# COMMAND: Generate collaborative recommendations

def get_collaborative_recommendations(
    customer_id,
    top_similar_users=10,
    top_n=10
):

    if customer_id not in customer_item_matrix.index:
        return pd.DataFrame()

    similar_customers = get_similar_customers(
        customer_id=customer_id,
        top_n=top_similar_users
    )

    if similar_customers.empty:
        return pd.DataFrame()

    # Products already interacted with by target customer
    customer_products = set(
        customer_product_scores.loc[
            customer_product_scores["customer_id"] == customer_id,
            "crop_name"
        ]
    )

    recommendations = []

    for _, row in similar_customers.iterrows():

        similar_customer_id = row["customer_id"]
        similarity = row["similarity"]

        # Get products interacted with by similar customer
        similar_customer_products = (
            customer_product_scores[
                customer_product_scores["customer_id"]
                == similar_customer_id
            ]
        )

        for _, product_row in similar_customer_products.iterrows():

            product = product_row["crop_name"]
            interaction_score = product_row[
                "interaction_score"
            ]

            # Do not recommend products the target
            # customer has already interacted with
            if product in customer_products:
                continue

            score = similarity * interaction_score

            recommendations.append({
                "customer_id": customer_id,
                "crop_name": product,
                "similar_customer_id": similar_customer_id,
                "similarity": similarity,
                "interaction_score": interaction_score,
                "score": score
            })

    if not recommendations:
        return pd.DataFrame()

    recommendations = pd.DataFrame(
        recommendations
    )

    # A product may be recommended by several
    # similar customers. Aggregate its evidence.
    product_recommendations = (
        recommendations
        .groupby(
            ["customer_id", "crop_name"],
            as_index=False
        )
        .agg(
            score=("score", "sum"),
            max_similarity=("similarity", "max"),
            supporting_customers=(
                "similar_customer_id",
                "nunique"
            )
        )
    )

    product_recommendations = (
        product_recommendations
        .sort_values(
            by=[
                "score",
                "max_similarity",
                "supporting_customers"
            ],
            ascending=False
        )
        .head(top_n)
        .reset_index(drop=True)
    )

    return product_recommendations

In [ ]:
# COMMAND: Test collaborative recommendations

customer_id = 1

print("Target customer:", customer_id)

print("\nPreviously interacted crops:")
customer_products = customer_product_scores.loc[
    customer_product_scores["customer_id"] == customer_id
].sort_values(
    "interaction_score",
    ascending=False
)

display(customer_products)

recommendations = get_collaborative_recommendations(
    customer_id=customer_id,
    top_similar_users=10,
    top_n=10
)

print("\nCollaborative recommendations:")

if recommendations.empty:
    print("No recommendations generated.")
else:
    display(recommendations)

## 9. Create Temporary Crop and Listing IDs

The final project integration requires:

`customer_id, listing_id, crop_id, crop_name, score, recommendation_type, reason`

The synthetic dataset does not have real database IDs, so temporary mappings are used for this notebook.

When the real database is connected, use the actual `crop` and `seller_listing` tables.

In [ ]:
# COMMAND: Create temporary crop/listing mapping

crop_mapping = pd.DataFrame({
    "crop_id": range(1, len(crops) + 1),
    "crop_name": crops
})

listing_mapping = crop_mapping.copy()

listing_mapping["listing_id"] = (
    listing_mapping["crop_id"] + 100
)

listing_mapping = listing_mapping[
    [
        "listing_id",
        "crop_id",
        "crop_name"
    ]
]

display(listing_mapping)

In [ ]:
# COMMAND: Generate final common integration output

def generate_final_collaborative_output(
    customer_id,
    top_similar_users=10,
    top_n=10
):

    recommendations = get_collaborative_recommendations(
        customer_id=customer_id,
        top_similar_users=top_similar_users,
        top_n=top_n
    )

    output_columns = [
        "customer_id",
        "listing_id",
        "crop_id",
        "crop_name",
        "score",
        "recommendation_type",
        "reason"
    ]

    if recommendations.empty:
        return pd.DataFrame(
            columns=output_columns
        )

    recommendations = recommendations.merge(
        listing_mapping,
        on="crop_name",
        how="left"
    )

    recommendations["recommendation_type"] = (
        "COLLABORATIVE"
    )

    recommendations["reason"] = (
        "Similar customers bought/interacted with this crop"
    )

    final_output = recommendations[
        output_columns
    ].copy()

    return final_output

In [ ]:
# COMMAND: Test final integration output

customer_id = 1

final_recommendations = (
    generate_final_collaborative_output(
        customer_id=customer_id,
        top_similar_users=10,
        top_n=5
    )
)

display(final_recommendations)

## 10. Test Multiple Customers

In [ ]:
# COMMAND: Generate recommendations for multiple customers

test_customers = [1, 2, 3, 4, 5]

all_test_recommendations = []

for customer_id in test_customers:

    result = generate_final_collaborative_output(
        customer_id=customer_id,
        top_similar_users=10,
        top_n=5
    )

    if not result.empty:
        all_test_recommendations.append(result)

if all_test_recommendations:

    all_test_recommendations = pd.concat(
        all_test_recommendations,
        ignore_index=True
    )

else:

    all_test_recommendations = pd.DataFrame(
        columns=[
            "customer_id",
            "listing_id",
            "crop_id",
            "crop_name",
            "score",
            "recommendation_type",
            "reason"
        ]
    )

display(all_test_recommendations)

In [ ]:
# COMMAND: Verify final output format

expected_columns = [
    "customer_id",
    "listing_id",
    "crop_id",
    "crop_name",
    "score",
    "recommendation_type",
    "reason"
]

assert list(
    all_test_recommendations.columns
) == expected_columns

if not all_test_recommendations.empty:
    assert (
        all_test_recommendations["recommendation_type"]
        .eq("COLLABORATIVE")
        .all()
    )

print("Final output format validation passed.")
print("Columns:", list(all_test_recommendations.columns))

## 11. Save Outputs

The following files are saved for testing/integration:

- `collaborative_interaction_data.csv`
- `collaborative_customer_item_matrix.csv`
- `collaborative_similarity_matrix.csv`
- `collaborative_recommendations.csv`

The final recommendation file follows the common output format expected by the integration notebook.

In [ ]:
# COMMAND: Save outputs

interaction_data.to_csv(
    "collaborative_interaction_data.csv",
    index=False
)

customer_item_matrix.to_csv(
    "collaborative_customer_item_matrix.csv"
)

customer_similarity.to_csv(
    "collaborative_similarity_matrix.csv"
)

all_test_recommendations.to_csv(
    "collaborative_recommendations.csv",
    index=False
)

print("All Collaborative Filtering output files saved successfully.")

## 12. Optional Supabase Connection

This section is **not required for the synthetic-data prototype**.

When the real project database is ready, the interaction data should be obtained from the project tables described in the specification:

- `user_interaction`
- `customer_order`
- `order_item`
- `seller_listing`
- `crop`

Then create the same `customer_id`, `crop_name`, and interaction-score representation before running the recommendation pipeline.

Do not put a Supabase service-role key in a shared/public Colab notebook.

In [ ]:
# COMMAND: Optional Supabase connection template

# from supabase import create_client

# SUPABASE_URL = "YOUR_SUPABASE_URL"
# SUPABASE_KEY = "YOUR_SUPABASE_ANON_KEY"

# supabase = create_client(
#     SUPABASE_URL,
#     SUPABASE_KEY
# )

# After connecting to your real tables,
# replace the synthetic interaction_data with
# the corresponding real interaction records.

## Final Pipeline

```text
Customer interactions
        ↓
Interaction scores
        ↓
Customer × Crop matrix
        ↓
Cosine similarity
        ↓
Find similar customers
        ↓
Products interacted with by similar customers
        ↓
Remove products already interacted with
        ↓
Rank recommendations
        ↓
Common integration output
```

### Final output

```text
customer_id
listing_id
crop_id
crop_name
score
recommendation_type
reason
```

For this section, **User-Based Collaborative Filtering** is the recommendation approach and **cosine similarity** is the similarity measure used to compare customer interaction patterns.

The notebook does not include FP-Growth or Buy Again logic.